In [1]:
import pickle
import os
import json
import glob
import gc
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from darts import TimeSeries
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from darts.models import TFTModel
from darts.utils.likelihood_models import (
    NegativeBinomialLikelihood,
    PoissonLikelihood,
)
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.preprocessing import OrdinalEncoder

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


In [2]:
#Loading the best checkpoint 
from darts.models import TFTModel

MODEL_NAME = 'daily_tft_negbin_scooters_2026-09-15_20_02_52'
loaded_model = TFTModel.load_from_checkpoint(model_name=MODEL_NAME,best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

Model loaded successfully with all dimensions intact!


In [3]:
#Loading the CACHE_DIR
BASE = os.getcwd()
CACHE_DIR       = os.path.join(os.getcwd(), "series_cache")


#Loading the ROLES Json : This helps to avoid hardcoding the variables
with open(os.path.join(BASE,"column_roles.json")) as f:
    ROLES = json.load(f)

time_col,group_col,target_col = ROLES["time_col"],ROLES["group_col"],ROLES["target_col"]

FREQ = ROLES["freq"]
static_covariates = ROLES["static_covariates"]

FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END = pd.Timestamp(ROLES["forecast_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1

safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

In [4]:
# ---------------- SEQUENCES ----------------
class History(collections.abc.Sequence):
    """Series history to forecast from. Reads the cached 'val' split,
    which ends on the day before FORECAST_START."""

    def __init__(self, keys, statics):
        self.keys, self.statics = keys, statics

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z["val_sales"], str(z["val_start"])
        return TimeSeries.from_times_and_values(
            pd.date_range(start, periods=len(sales), freq=FREQ),
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=self.statics[i],
        )


class SharedCov(collections.abc.Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        return self.cov

In [5]:
print(f"Forecast: {FORECAST_START.date()} -> {FORECAST_END.date()} ({HORIZON} days)")

Forecast: 2026-09-01 -> 2026-12-07 (98 days)


In [6]:
BATCH_SIZE = 512        # raise until GPU memory complains
BLOCK      = 5000       # series per block, written to disk as it goes
LIMIT      = None       # set to e.g. 2000 for a quick smoke test

In [7]:
with open(os.path.join(CACHE_DIR, "manifest.json")) as f:
    manifest = json.load(f)

static_df = pd.read_parquet(os.path.join(CACHE_DIR, "static_covariates.parquet"))

# same dead-series filter as training, so indices stay aligned with static_df
keep = []
for k in manifest["series_keys"]:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        keep.append(z["train_sales"].sum() > 0)

series_keys = [k for k, m in zip(manifest["series_keys"], keep) if m]
has_val     = [h for h, m in zip(manifest["has_val"],     keep) if m]
static_df   = static_df.loc[keep].reset_index(drop=True)[static_covariates].astype(str)

keys = [k for k, h in zip(series_keys, has_val) if h]
idxs = [i for i, h in enumerate(has_val) if h]
if LIMIT:
    keys, idxs = keys[:LIMIT], idxs[:LIMIT]

print(f"Series to forecast: {len(keys):,}")
if len(keys) < len(series_keys):
    print(f"  ({len(series_keys) - len(keys):,} lack enough history and are skipped)")

with open(os.path.join(CACHE_DIR, "static_cov_transformer.pkl"), "rb") as f:
    sc_transformer = pickle.load(f)

SHARED_COV = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
if SHARED_COV.end_time() < FORECAST_END:
    raise ValueError(f"Covariates end {SHARED_COV.end_time().date()}, need {FORECAST_END.date()}")

# encode static covariates once, not once per series inside the loader
dummy_t = pd.date_range("2000-01-01", periods=2, freq="D")
statics = []
for i in idxs:
    t = TimeSeries.from_times_and_values(
        dummy_t, np.zeros((2, 1), dtype=np.float32), columns=[target_col],
        static_covariates=static_df.iloc[[i]].reset_index(drop=True),
    )
    statics.append(sc_transformer.transform(t).static_covariates)

# load BEST weights -- model.predict() on an in-memory model uses the LAST
# epoch, which with patience=5 is 5 epochs past what early stopping picked
model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
print(f"Loaded best checkpoint: {MODEL_NAME}")


Series to forecast: 31,931
Loaded best checkpoint: daily_tft_negbin_scooters_2026-09-15_20_02_52


In [8]:
# NOTE: new directory on purpose. The previous run's block parquets were
# written with the broken summarise() (it read the NegBin params as (mu, alpha)
# when Darts actually returns (r, p)). Writing to a fresh folder means the
# "exists, skipping" resume logic cannot silently reuse those wrong values, and
# nothing is deleted -- the old output stays around for comparison.
OUT_DIR = os.path.join(os.getcwd(), "predictions_2026_iteration_2_negbin_fixed")

In [9]:
from scipy import stats

In [13]:
# ---------------- FORECAST ----------------
# Darts' NegativeBinomialLikelihood.predict_likelihood_parameters() returns
# (r, p), NOT (mu, alpha). From the Darts source:
#
#     r = 1 / alpha
#     p = r / (mu + r)
#     return torch.cat([r, p], dim=-1)
#
# so the component names come back as "<target>_r" and "<target>_p" -- which is
# exactly what this notebook printed last run. Reading column 0 as the mean is
# wrong, and wrong in a volume-dependent way: for a quiet day (true mean ~1.2)
# r happens to land near 1.1, but for a festive day (true mean ~45) r is ~5.
# Peaks get crushed to roughly a tenth of their real size while quiet days come
# out about right -- which is precisely the "September too high, November hollow"
# shape we were chasing.
#
# Inverting correctly: mean = r * (1 - p) / p, and scipy's nbinom(n=r, p=p) uses
# the same convention, so the quantiles are a direct ppf call on (r, p).

QUANTILES = [35,40,50,60]


def summarise(p_ts):
    """NegBin params (r, p) from Darts -> mean and quantiles, closed form."""
    v  = p_ts.values(copy=False)
    r  = np.clip(v[:, 0].astype(np.float64), 1e-9, None)
    pr = np.clip(v[:, 1].astype(np.float64), 1e-9, 1.0 - 1e-9)

    out = {"PRED_MEAN": r * (1.0 - pr) / pr}
    for q_int in QUANTILES:
        out[f"PRED_Q{q_int}"] = stats.nbinom.ppf(q_int / 100.0, r, pr)
    return out


EXPECTED_PARAMS = [f"{target_col}_r", f"{target_col}_p"]
EXPECTED_COLS   = {"PRED_MEAN", *(f"PRED_Q{q}" for q in QUANTILES)}

os.makedirs(OUT_DIR, exist_ok=True)
n_blocks = (len(keys) + BLOCK - 1) // BLOCK
paths, t0, checked = [], datetime.now(), False

for b in range(n_blocks):
    lo, hi = b * BLOCK, min((b + 1) * BLOCK, len(keys))
    path = os.path.join(OUT_DIR, f"{MODEL_NAME}_block_{b:04d}.parquet")
    paths.append(path)

    # resume, but only if the cached block has the columns this code produces.
    # Existence alone is not enough: a block written by an older summarise()
    # silently survives and poisons the concatenated output at the end.
    if os.path.exists(path):
        cached_cols = set(pd.read_parquet(path, columns=[]).columns)
        if EXPECTED_COLS.issubset(cached_cols):
            print(f"[{b+1}/{n_blocks}] exists with current schema, skipping")
            continue
        print(f"[{b+1}/{n_blocks}] exists but schema is stale "
              f"(missing {sorted(EXPECTED_COLS - cached_cols)}) -- recomputing")

    preds = model.predict(
        n=HORIZON,
        series=History(keys[lo:hi], statics[lo:hi]),
        future_covariates=SharedCov(SHARED_COV, hi - lo),
        predict_likelihood_parameters=True,   # 1 pass instead of num_samples passes
        num_samples=1,
        batch_size=BATCH_SIZE,
        verbose=False,
    )

    if not checked:   # fail fast if dates or params are not what summarise() assumes
        assert preds[0].start_time() == FORECAST_START, \
            f"forecast starts {preds[0].start_time().date()}, expected {FORECAST_START.date()}"
        got = list(preds[0].components)
        assert got == EXPECTED_PARAMS, (
            f"likelihood parameters are {got}, expected {EXPECTED_PARAMS}. "
            f"summarise() decodes (r, p) -- do not run it on anything else."
        )
        print(f"  params: {got}  (decoded as r, p)")
        checked = True

    pd.concat(
        [pd.DataFrame({group_col: k, time_col: p_ts.time_index, **summarise(p_ts)})
         for k, p_ts in zip(keys[lo:hi], preds)],
        ignore_index=True,
    ).to_parquet(path, index=False)

    el = (datetime.now() - t0).total_seconds()
    print(f"[{b+1}/{n_blocks}] {hi:,}/{len(keys):,} | {hi/el:,.0f} series/s | "
          f"ETA {(len(keys)-hi)/(hi/el)/60:.0f} min")

    del preds
    gc.collect()


# ---------------- OUTPUT ----------------
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
out = os.path.join(OUT_DIR, f"{MODEL_NAME}_predictions.parquet")
df.to_parquet(out, index=False)

print(f"\nWritten -> {out}")
print(f"  rows: {len(df):,} | elapsed: {(datetime.now()-t0).total_seconds()/60:.1f} min")
for c in ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]:
    print(f"  {c}: {df[c].sum():,.0f} ({df[c].sum()/1e5:.2f} lacs)")

Using bfloat16 Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


  params: ['NET_SALES_r', 'NET_SALES_p']  (decoded as r, p)


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[1/7] 5,000/31,931 | 95 series/s | ETA 5 min


...

[4/7] 20,000/31,931 | 82 series/s | ETA 2 min


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[5/7] 25,000/31,931 | 79 series/s | ETA 1 min


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[6/7] 30,000/31,931 | 80 series/s | ETA 0 min
[7/7] 31,931/31,931 | 81 series/s | ETA 0 min

Written -> c:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling\predictions_2026_iteration_2_negbin_fixed\daily_tft_negbin_scooters_2026-09-15_20_02_52_predictions.parquet
  rows: 3,129,238 | elapsed: 6.7 min
  PRED_MEAN: 5,855,276 (58.55 lacs)
  PRED_Q35: 942,628 (9.43 lacs)
  PRED_Q40: 1,248,262 (12.48 lacs)
  PRED_Q50: 2,025,757 (20.26 lacs)
  PRED_Q60: 3,079,715 (30.80 lacs)


In [14]:
# Read the predictions back from the path this run just wrote,
# rather than a hardcoded absolute path to the previous (wrong) output.
parquet_pred = pd.read_parquet(out)
print(f"{len(parquet_pred):,} rows from {out}")

3,129,238 rows from c:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling\predictions_2026_iteration_2_negbin_fixed\daily_tft_negbin_scooters_2026-09-15_20_02_52_predictions.parquet


In [ ]:
# import shutil 
# shutil.rmtree(OUT_DIR)

In [29]:
parquet_pred.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,PRED_MEAN,PRED_Q60,PRED_Q70,PRED_Q72,PRED_Q80,PRED_Q85
0,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-01,1.136725,1.0,2.0,2.0,2.0,2.0
1,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-02,1.218015,1.0,2.0,2.0,2.0,2.0
2,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-03,1.186923,1.0,2.0,2.0,2.0,2.0
3,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-04,1.345593,1.0,2.0,2.0,2.0,3.0
4,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-05,1.176025,1.0,2.0,2.0,2.0,2.0


In [15]:
parquet_pred["CAL_DATE"] = pd.to_datetime(parquet_pred["CAL_DATE"])
parquet_pred["MONTH_NAME"] = parquet_pred["CAL_DATE"].dt.month_name()

In [ ]:
# The previous, incorrectly-decoded output is still on disk under
#   predictions_2026_iteration_2_wo_cal_att
# Delete it once you have confirmed this run looks right -- keeping it around
# invites someone reading the wrong parquet later.
#
# import shutil
# shutil.rmtree(os.path.join(os.getcwd(), "predictions_2026_iteration_2_wo_cal_att"))

In [16]:
# ---------------- SHAPE CHECK ----------------
# The total being plausible means little on its own -- the failure mode here was
# always the SHAPE. Compare the predicted month split against what the last three
# festive seasons actually did.
#
# The historical shares below are calendar-drift corrected: each past year was
# mapped onto 2026's month boundaries in festive-offset terms (2026 has N=Oct 11,
# D=Nov 8, so Sep = N-40..N-11, Oct = N-10..D-8, Nov = D-7..D+22, Dec = D+23..D+29)
# using the real anchors from Festive_Data_for_Daily_forecasting.csv. Derived from
# the 07_festive_profile.py export, which covers a ~8% subset of series -- so treat
# these as shape guidance, not exact population shares.
HIST_SHARE = {        # % of the Sep 1 - Dec 7 total
    "September": {"2023": 10.9, "2024":  4.8, "2025":  7.4, "mean":  7.7},
    "October":   {"2023": 16.8, "2024": 16.0, "2025": 19.1, "mean": 17.3},
    "November":  {"2023": 71.2, "2024": 78.2, "2025": 71.3, "mean": 73.6},
    "December":  {"2023":  1.1, "2024":  1.0, "2025":  2.1, "mean":  1.4},
}

parquet_pred["CAL_DATE"]  = pd.to_datetime(parquet_pred["CAL_DATE"])
parquet_pred["MONTH_NAME"] = parquet_pred["CAL_DATE"].dt.month_name()

ORDER = ["September", "October", "November", "December"]
COLS  = ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]

split = (parquet_pred.groupby("MONTH_NAME")[COLS].sum()
         .reindex(ORDER))

print("TOTALS BY MONTH")
print(split.round(0).to_string())

print("\nSHARE OF WINDOW (%) -- predicted vs actual history")
share = 100 * split / split.sum()
share.insert(0, "HIST_mean", [HIST_SHARE[m]["mean"] for m in ORDER])
print(share.round(1).to_string())

gap = share["PRED_MEAN"] - share["HIST_mean"]
print("\nPRED_MEAN share minus historical mean share (pp):")
print(gap.round(1).to_string())
print("\nNote: December is only 7 days (Dec 1-7), so a small share is expected.")

TOTALS BY MONTH
            PRED_MEAN  PRED_Q35  PRED_Q40  PRED_Q50   PRED_Q60
MONTH_NAME                                                    
September   2340500.0  427401.0  556978.0  882710.0  1311426.0
October     1844478.0  299957.0  392461.0  622990.0   938084.0
November    1282762.0  163897.0  226499.0  394431.0   632132.0
December     387536.0   51373.0   72324.0  125626.0   198073.0

SHARE OF WINDOW (%) -- predicted vs actual history
            HIST_mean  PRED_MEAN  PRED_Q35  PRED_Q40  PRED_Q50  PRED_Q60
MONTH_NAME                                                              
September         7.7       40.0      45.3      44.6      43.6      42.6
October          17.3       31.5      31.8      31.4      30.8      30.5
November         73.6       21.9      17.4      18.1      19.5      20.5
December          1.4        6.6       5.4       5.8       6.2       6.4

PRED_MEAN share minus historical mean share (pp):
MONTH_NAME
September    32.3
October      14.2
November    -51.7


In [34]:
import os, json
import numpy as np
import pandas as pd
from darts import TimeSeries

In [35]:
BASE      = os.getcwd()
CACHE_DIR = os.path.join(os.getcwd(), "series_cache")

In [36]:
FC_START = pd.Timestamp("2026-09-01")
FC_END   = pd.Timestamp("2026-12-07")

with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
time_col = ROLES["time_col"]

N_BLOCK = [f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
D_BLOCK = [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]


def frame_from_pickle(path):
    ts = TimeSeries.from_pickle(path)
    df = ts.to_dataframe().reset_index()
    df.columns = [time_col] + list(ts.components)
    return df


# ------------------------------------------------------------------ load both
sources = {}

pkl = os.path.join(CACHE_DIR, "shared_cov.pkl")
if os.path.exists(pkl):
    sources["shared_cov.pkl (what inference uses)"] = frame_from_pickle(pkl)
else:
    print("shared_cov.pkl NOT FOUND -- inference would have rebuilt from parquet.")

pq = os.path.join(BASE, "shared_calendar.parquet")
if os.path.exists(pq):
    d = pd.read_parquet(pq)
    d[time_col] = pd.to_datetime(d[time_col])
    sources["shared_calendar.parquet (source)"] = d


for label, cal in sources.items():
    print("=" * 70)
    print(label)
    print("=" * 70)

    cal[time_col] = pd.to_datetime(cal[time_col])
    print(f"Date range : {cal[time_col].min().date()} -> {cal[time_col].max().date()}")
    print(f"Columns    : {len(cal.columns) - 1}")

    if cal[time_col].max() < FC_END:
        print(f"\n*** FAIL: calendar ends before {FC_END.date()}. "
              f"Covariates are missing for part of the forecast. ***")

    win = cal[(cal[time_col] >= FC_START) & (cal[time_col] <= FC_END)]
    print(f"Rows in Sep 1 - Dec 7 2026: {len(win)} (expected 98)")

    if len(win) == 0:
        print("\n*** FAIL: no calendar rows in the forecast window at all. ***\n")
        continue

    # --- NaN check: NaNs silently poison the whole forecast ------------------
    nan_cols = [c for c in win.columns if c != time_col and win[c].isna().any()]
    if nan_cols:
        print(f"\n*** FAIL: {len(nan_cols)} columns contain NaN in the window: "
              f"{nan_cols[:8]} ***")
    else:
        print("NaN check : clean")

    # --- the festive blocks --------------------------------------------------
    for name, block, expect in [("N block", N_BLOCK, 27), ("D block", D_BLOCK, 10)]:
        present = [c for c in block if c in win.columns]
        missing = [c for c in block if c not in win.columns]

        print(f"\n{name}: {len(present)}/{len(block)} columns present")
        if missing:
            print(f"  MISSING COLUMNS: {missing}")

        live = {c: int((win[c] != 0).sum()) for c in present}
        n_live = sum(1 for v in live.values() if v > 0)
        print(f"  columns with a non-zero day in the window: {n_live}/{expect}")

        if n_live == 0:
            print("  *** FAIL: the entire block is zero across the forecast window. ***")
            print("      The model is flying blind on this festival. This is the bug.")
        elif n_live < expect:
            dead = [c for c, v in live.items() if v == 0]
            print(f"  *** PARTIAL: these are all-zero: {dead}")
        else:
            print("  OK: every column fires at least once.")

        # each flag should fire on exactly one day
        multi = {c: v for c, v in live.items() if v > 1}
        if multi:
            print(f"  NOTE: fire on >1 day (expected 1 each): {multi}")

    # --- where does each anchor land -----------------------------------------
    for anchor in ["N", "D"]:
        if anchor in win.columns:
            hits = win.loc[win[anchor] != 0, time_col]
            print(f"\n'{anchor}' day-0 in window: "
                  f"{[d.date().isoformat() for d in hits] or 'NONE'}")

    # --- days with no festive signal at all ----------------------------------
    fest = [c for c in (N_BLOCK + D_BLOCK) if c in win.columns]
    if fest:
        blank = win.loc[(win[fest] == 0).all(axis=1), time_col]
        print(f"\nDays in the window with NO festive flag: {len(blank)}/98")
        if len(blank):
            print(f"  first: {blank.min().date()}   last: {blank.max().date()}")
            sep = blank[blank.dt.month == 9]
            print(f"  of those, {len(sep)} fall in September "
                  f"({sep.min().date() if len(sep) else '-'} to "
                  f"{sep.max().date() if len(sep) else '-'})")
            print("  On these days the model has no festive signal and can only")
            print("  extrapolate the baseline from the encoder.")
    print()


# ------------------------------------------------------------------ compare
if len(sources) == 2:
    print("=" * 70)
    print("DO THE TWO ARTIFACTS AGREE?")
    print("=" * 70)
    a, b = list(sources.values())
    ca = set(a.columns) - {time_col}
    cb = set(b.columns) - {time_col}
    if ca != cb:
        print(f"*** Column sets differ. only in pkl: {sorted(ca-cb)[:8]} | "
              f"only in parquet: {sorted(cb-ca)[:8]} ***")
    else:
        wa = a[(a[time_col] >= FC_START) & (a[time_col] <= FC_END)].set_index(time_col).sort_index()
        wb = b[(b[time_col] >= FC_START) & (b[time_col] <= FC_END)].set_index(time_col).sort_index()
        cols = sorted(ca)
        same = np.allclose(wa[cols].to_numpy(float), wb[cols].to_numpy(float), equal_nan=True)
        print("Values identical in the forecast window:", same)
        if not same:
            print("*** The pickle inference uses differs from the source parquet.")
            print("    Rebuild shared_cov.pkl. ***")


shared_cov.pkl (what inference uses)
Date range : 2023-04-01 -> 2026-12-07
Columns    : 57
Rows in Sep 1 - Dec 7 2026: 98 (expected 98)
NaN check : clean

N block: 27/27 columns present
  columns with a non-zero day in the window: 27/27
  OK: every column fires at least once.

D block: 10/10 columns present
  columns with a non-zero day in the window: 10/10
  OK: every column fires at least once.

'N' day-0 in window: ['2026-10-11']

'D' day-0 in window: ['2026-11-08']

Days in the window with NO festive flag: 63/98
  first: 2026-09-01   last: 2026-12-07
  of those, 25 fall in September (2026-09-01 to 2026-09-25)
  On these days the model has no festive signal and can only
  extrapolate the baseline from the encoder.

shared_calendar.parquet (source)
Date range : 2023-04-01 -> 2026-12-07
Columns    : 57
Rows in Sep 1 - Dec 7 2026: 98 (expected 98)
NaN check : clean

N block: 27/27 columns present
  columns with a non-zero day in the window: 27/27
  OK: every column fires at least once.